In [1]:
#defining a zero-lag ema
def zlema(series, period):
    ema1 = talib.EMA(series, period)
    ema2 = talib.EMA(ema1, period)
    return 2 * ema1 - ema2
#implementig vectorized operation to define crossovers 
def vectorized_crossover(series1, series2):
    return (series1 > series2) & (series1.shift(1) < series2.shift(1))
    
def vectorized_crossunder(series1, series2):
    return (series1 < series2) & (series1.shift(1) > series2.shift(1))
    
#defining a zero-lag macd, cond_buy is defined whenever macd line and signal line crosses under zero line
#simmetrically for cond_sell
def macd_impl(df):
    df['fast_period'] = zlema(df['Close'], 12)
    df['slow_period'] = zlema(df['Close'], 26)
    df['macd'] = df['fast_period'] - df['slow_period']
    df['signal'] = zlema(df['macd'], 9)
    df['hist'] = df['macd'] - df['signal']
    df['atr'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    
    cond_buy = (
    vectorized_crossover(df['macd'], df['signal']) & 
    (df['macd'] < 0)
    )
    
    cond_sell = (
    vectorized_crossunder(df['macd'], df['signal']) & 
    (df['macd'] > 0)
    )
    conditions = [cond_buy, cond_sell]
    choices    = [1, -1]
#Trade_Direction will translate cond_buy and cond_sell into 1 and -1
    df['Trade_Direction'] = np.select(conditions, choices, default=0)
    df['stop_loss'] = np.where(
        df['Trade_Direction'] > 0,
        df['Close'] - (df['atr'] * 2.5),
        df['Close'] + (df['atr'] * 2.5)
         )
    #df['Trade_Direction'] = df['Trade_Direction'].shift(-1)
    return df

In [2]:
#this function will aling the trade result with the trigger candle
def implement_trades(df, df_results):
    
    df['Trades'] = np.nan
    
    df.iloc[df_results['EntryBar'].values, df.columns.get_loc('Trades')] = df_results['PnL'].values
    
    df.loc[df['Trades'] > 0, 'Trades'] = 1  #PnL > 0 win trade
    df.loc[df['Trades'] < 0, 'Trades'] = 0 #PnL <0 loss trade
    df['Trades'] = df['Trades']
    return df

In [3]:

def price_features(df):

    df['price_slope'] = talib.LINEARREG_SLOPE(df['Close'], timeperiod = 20)

    don_upper = talib.MAX(df['High'], timeperiod = 20)
    don_lower = talib.MIN(df['Low'], timeperiod = 20)
    df['donchain_pos'] = (df['Close'] - don_lower) / (don_upper - don_lower)
    
    return df

In [4]:
def indicators_features(df):
    
    df['sar'] = talib.SAR(df['High'], df['Low'])
    df['Rsi_9'] = talib.RSI(df['Close'], timeperiod = 9)
    df['Rsi_14'] = talib.RSI(df['Close'], timeperiod = 14)
    df['OBV'] = talib.OBV(df['Close'], df['Volume'])
    
    return df

In [5]:
def ema_features(df):
    
    ema_50 = talib.EMA(df['Close'], 50)
    ema_200 = talib.EMA(df['Close'], 200)
    
    df['ema_50_slope'] = talib.LINEARREG_SLOPE(ema_50, timeperiod = 50)
    df['ema_200_slope'] = talib.LINEARREG_SLOPE(ema_200, timeperiod = 50)
    
    df['ema_gap'] = (ema_50 - ema_200) / ema_200 * 100
    
    df['distance_from_ema_50'] = (df['Close'] - ema_50).abs() / df['Close']
    df['distance_from_ema_200'] = (df['Close'] - ema_200).abs() / df['Close']

    return df

In [6]:
import pandas as pd
import talib
import numpy as np
from backtesting import Strategy, Backtest

/home/matteozamaro/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/matteozamaro/venv/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [7]:
df = pd.read_csv('btcusdt_spot_last_3years_italy.csv')

In [8]:
df.columns = df.columns.str.capitalize()
macd_impl(df)
price_features(df)
indicators_features(df)
ema_features(df)
df.dropna(inplace = True)

In [9]:
class trade_status_strategy(Strategy):
    
    def init(self):
        #empty function in which we should put internal calcs for indicators, required even if empty
        pass

    def next(self):

        price = self.data.Close[-1]
        current_atr = self.data.atr[-1]
        trade_signal = self.data.Trade_Direction[-1]

        if not self.position:
            if trade_signal == -1:  # Short Signal
                # Calculate levels based on the moment of the signal
                sl_price = price + (current_atr * 2.5)
                tp_price = price - (current_atr * 5)
                
                # Execute at current close (or next open) using these static values
                self.sell(size=0.01, sl=sl_price, tp=tp_price)

            elif trade_signal == 1:  # Long Signal
                sl_price = price - (current_atr * 2.5)
                tp_price = price + (current_atr * 5)
                
                self.buy(size=0.01, sl=sl_price, tp=tp_price)

In [10]:
bt = Backtest(
    df,
    trade_status_strategy,
    cash=100000000,
    exclusive_orders=True,
    trade_on_close=True
)
results = bt.run()
results

/home/matteozamaro/venv/lib/python3.11/site-packages/backtesting/backtesting.py:1212: FutureWarning: Index.is_numeric is deprecated. Use pandas.api.types.is_any_real_numeric_dtype instead
  (data.index.is_numeric() and
/tmp/ipykernel_3686/1750268398.py:1: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(
/tmp/ipykernel_3686/1750268398.py:8: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  results = bt.run()


Start                                   248.0
End                                  315343.0
Duration                             315095.0
Exposure Time [%]                    92.17318
Equity Final [$]              101546210.59543
Equity Peak [$]               101634813.44511
Return [%]                            1.54621
Buy & Hold Return [%]                436.1063
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Alpha [%]                             1.04792
Beta                                  0.00114
Max. Drawdown [%]                    -0.32218
Avg. Drawdown [%]                    -0.00942
Max. Drawdown Duration                57284.0
Avg. Drawdown Duration              433.30663
# Trades                               5537.0
Win Rate [%]                         34.83836
Best Trade [%]                    

In [11]:
df_results = results['_trades']
implement_trades(df, df_results)
df_trades = df.loc[(df['Trades'] == 0) | (df['Trades'] == 1)]

In [12]:
df.to_csv('btc_3y_with_features.csv')
df_trades.to_csv('btc_trades_features_02.csv')